# 03 · Popularity fallback (historical CTR)

**Role in the pipeline:** base scorer #3. Neither UBCF nor CBF can score a user with **zero clicks** so a smoothed historical CTR is the fallback signal. Saved as `scores_pop_{phase}` in the shared output format.

**Frozen training window (default, same as UBCF and CBF).** For each phase, clicks and exposures are counted **only from that phase's build days** (`common.PHASES`):

$$\text{ctr}(a) = \frac{\text{clicks}_{\text{build}}(a) + \alpha\,\mu}{\text{exposures}_{\text{build}}(a) + \alpha}$$

Nothing from the validation/test day is used, so popularity stops learning at the same point as the other two models. μ = mean CTR of the build window (= training set for val/test), so an article never shown during training gets exactly the training mean – the contract's missing-value rule – with `available = 0`. α is tuned on validation.

**Optional online variant** (`ONLINE_CTR = True`): counts every impression strictly before *t*, including earlier impressions on the val/test day itself. That is what a live system would do, but it learns from test-day clicks, so it is **not** comparable with UBCF/CBF. Report it only as a separately labelled sensitivity result.

In [6]:
import numpy as np
import pandas as pd
import common as C
import importlib
importlib.reload(C)

ONLINE_CTR = False          # keep False for the main results (frozen window, contract-consistent)
MODEL_NAME = "pop_online" if ONLINE_CTR else "pop"

beh = C.load_behaviors()
cand = C.load_candidates(beh)

## Clicks / exposures per article

In [7]:
def frozen_counts(build_days):
    """Clicks and exposures per article over the build window only."""
    b = cand[cand["date"].isin(build_days)]
    counts = b.groupby("news_id")["label"].agg(prior_clicks="sum", prior_exps="size")
    return counts, float(b["label"].mean())


def online_counts():
    """Clicks and exposures per (article, time) over all impressions strictly before t."""
    agg = (cand.groupby(["news_id", "time"], sort=False)["label"]
               .agg(clicks="sum", exps="size").reset_index()
               .sort_values(["news_id", "time"]))
    g = agg.groupby("news_id", sort=False)
    agg["prior_clicks"] = g["clicks"].cumsum() - agg["clicks"]   # exclusive of same timestamp
    agg["prior_exps"] = g["exps"].cumsum() - agg["exps"]
    return agg[["news_id", "time", "prior_clicks", "prior_exps"]]


ONLINE = online_counts() if ONLINE_CTR else None


def phase_frame(phase):
    cfg = C.PHASES[phase]
    log = C.build_click_log(beh, cand, cfg["build_days"])
    c = C.phase_candidates(cand, phase)
    c = C.add_cold_start_flags(c, beh, log, cfg["score_days"])
    counts, mu = frozen_counts(cfg["build_days"])
    if ONLINE_CTR:
        c = c.merge(ONLINE, on=["news_id", "time"], how="left")
    else:
        c = c.join(counts, on="news_id")
    c[["prior_clicks", "prior_exps"]] = c[["prior_clicks", "prior_exps"]].fillna(0)
    c["mu"] = mu
    c["available"] = (c["prior_exps"] > 0).astype(np.int8)
    c["pop_log_exps"] = np.log1p(c["prior_exps"])       # NOTE: check with group - partly a recency proxy
    return c


def pop_score(df, alpha):
    return (df["prior_clicks"] + alpha * df["mu"]) / (df["prior_exps"] + alpha)

## Tune smoothing α on validation

In [8]:
v = phase_frame("val")
print(f"training-window mean CTR μ = {v['mu'].iloc[0]:.4f} | "
      f"val candidates with no training exposures: {(v['available'] == 0).mean():.1%}")
runs = []
for alpha in (1, 5, 20, 100, 500):
    r = C.evaluate(v.assign(score=pop_score(v, alpha)), by_segment=False).iloc[0]
    runs.append({"alpha": alpha, **r.to_dict()})
tuning = pd.DataFrame(runs).sort_values("nDCG@10", ascending=False)
ALPHA = float(tuning.iloc[0]["alpha"])
print("best α:", ALPHA)
tuning.round(4)

training-window mean CTR μ = 0.0410 | val candidates with no training exposures: 29.2%
best α: 1.0


,alpha,impressions,MRR,nDCG@5,nDCG@10,AUC
0,1,30270.0,0.2739,0.2437,0.3060,0.5725
3,100,30270.0,0.2674,0.2421,0.3037,0.5734
1,5,30270.0,0.2685,0.2415,0.3037,0.5741
2,20,30270.0,0.2670,0.2415,0.3037,0.5739
4,500,30270.0,0.2662,0.2398,0.3020,0.5718


## Score all phases and save

In [9]:
for phase in C.PHASES:
    df = phase_frame(phase)
    df["score"] = pop_score(df, ALPHA)
    df["pop_ctr"] = df["score"]
    path = C.save_scores(df, MODEL_NAME, phase, extra_cols=["pop_ctr", "pop_log_exps"])
    print(f"{phase:12s} rows={len(df):,}  available={df['available'].mean():.1%} -> {path}")

stack_train  rows=1,231,673  available=79.2% -> /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4222/Project/bt4222-group12/models/2_UBCF + CBF/outputs/scores_pop_stack_train.parquet
val          rows=1,222,429  available=70.8% -> /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4222/Project/bt4222-group12/models/2_UBCF + CBF/outputs/scores_pop_val.parquet
test         rows=2,740,998  available=19.4% -> /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4222/Project/bt4222-group12/models/2_UBCF + CBF/outputs/scores_pop_test.parquet


## Standalone evaluation

In [10]:
res = C.standalone_eval(MODEL_NAME)
for ph, table in res.items():
    table = table[["impressions", "MRR", "nDCG@5", "nDCG@10", "AUC"]]
    print(f"\n=== Popularity ({'online' if ONLINE_CTR else 'frozen'}) · {ph} ===")
    print(table.round(4).to_string())
    table.to_csv(f"{C.OUT_DIR}/metrics_{MODEL_NAME}_{ph}.csv")


=== Popularity (frozen) · val ===
                                       impressions     MRR  nDCG@5  nDCG@10     AUC
All                                          30270  0.2737  0.2436   0.3060  0.5723
Warm users                                   30110  0.2738  0.2436   0.3060  0.5723
Cold users (0 clicks)                          160  0.2663  0.2333   0.3088  0.5783
Clicked article seen before                   8276  0.3584  0.3565   0.4143  0.7368
Clicked article new (no prior clicks)        21994  0.2419  0.2011   0.2652  0.5104

=== Popularity (frozen) · test ===
                                       impressions     MRR  nDCG@5  nDCG@10     AUC
All                                          73152  0.2470  0.2243   0.2894  0.5192
Warm users                                   71152  0.2469  0.2242   0.2893  0.5193
Cold users (0 clicks)                         2000  0.2499  0.2304   0.2947  0.5161
Clicked article seen before                   8137  0.5926  0.6105   0.6284  0.6960
Click